# Diagnostic — colab_11 CPT came back INERT (held-out drift 0.0057 << 0.05 gate)

colab_11's runtime is gone and the notebook is frozen; this is a standalone read-only diagnostic that works from the **saved artifacts on Drive** only. It does not re-run any part of the pipeline and produces no committed output.

The near-zero drift has three candidate explanations, which need different responses:

1. **Genuine null** — the LoRA update is honest but barely moves the pooled embedding.
2. **Undertrained** — attention-only r=8 over ~0.67 epoch is too weak a perturbation.
3. **Silent mechanical failure** — gradients never reached the adapter, or the merge never folded the adapter into the base, so the extractor embedded the unmodified base model.

This notebook rules #3 in or out **before** we touch any hyperparameter. Run one stage at a time.

- **Stage 1** — did the adapter weights move at all? (needs the adapter on Drive; seconds, CPU)
- **Stage 2** — does merging fold that update into the base weights? (re-downloads the base; a few min, CPU)
- **Stage 3** — plan only, flesh out if Stages 1-2 both pass.

In [ ]:
# Colab preinstalls torchao 0.10.0; PEFT's is_torchao_available() hard-raises
# below its 0.16.0 floor even though nothing here uses it -- remove it (same fix as colab_11 setup).
!pip uninstall -y torchao -q

from google.colab import drive
drive.mount('/content/drive')

ADAPTER_DIR = '/content/drive/MyDrive/ad-glia-fm-prep/geneformer/cpt_aggregated_seed0_adapter'

import os
print('adapter dir contents:')
for f in sorted(os.listdir(ADAPTER_DIR)):
    print(' ', f, os.path.getsize(os.path.join(ADAPTER_DIR, f)), 'bytes')

## Stage 1 — did the adapter weights move?

PEFT initializes LoRA's `B` matrix to exact zero, so at step 0 the adapter has literally no effect. If `lora_B` is still ~0 after training, gradients never reached it (frozen `requires_grad`, optimizer not stepping, etc.) — a mechanical failure, and the drift result is meaningless. If `lora_B` norms are clearly non-zero, training moved the adapter and we proceed to Stage 2.

In [ ]:
from safetensors.torch import load_file

sd = load_file(os.path.join(ADAPTER_DIR, 'adapter_model.safetensors'))

b_norms = []
for name, t in sd.items():
    tag = ''
    if 'lora_B' in name:
        b_norms.append(t.norm().item())
        tag = '  <- B (zero at init)'
    print(f'{name:70s} shape={str(tuple(t.shape)):14s} norm={t.norm().item():.6f} mean_abs={t.abs().mean().item():.6e}{tag}')

print()
if b_norms:
    import numpy as np
    b = np.array(b_norms)
    print(f'lora_B tensors: {len(b)} | norm min={b.min():.6f} median={np.median(b):.6f} max={b.max():.6f}')
    print('VERDICT:', 'B moved -> gradients flowed, go to Stage 2' if b.min() > 1e-6 else 'B still ~0 -> gradients never reached adapter (mechanical failure)')
else:
    print('no lora_B tensors found -- check adapter naming')

## Stage 2 — does merging fold the update into the base weights?

Re-download the exact base recorded in `adapter_config.json`, snapshot a couple of attention-projection weights, apply the adapter, `merge_and_unload()`, and diff. This reproduces colab_11 §6a's merge step. If the merged weights differ from the base by a real (if small) relative amount, the merge mechanism is sound and #3 is ruled out — leaving the genuine-null vs undertrained call. If the delta is exactly 0, the merge is a no-op and the extractor was embedding the base model.

In [ ]:
import json, torch
from transformers import BertForMaskedLM
from peft import PeftModel

cfg = json.load(open(os.path.join(ADAPTER_DIR, 'adapter_config.json')))
base_path = cfg['base_model_name_or_path']
print('base_model_name_or_path:', base_path)
print('target_modules:', cfg.get('target_modules'), '| r:', cfg.get('r'), '| alpha:', cfg.get('lora_alpha'))

# The adapter records the base as a local path from colab_11's (now-gone) runtime.
# Re-fetch just the V2-104M weights to that exact path at colab_11's pinned commit.
if not os.path.exists(os.path.join(base_path, 'config.json')):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='ctheodoris/Geneformer',
                      revision='04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5',
                      allow_patterns=['Geneformer-V2-104M/*'],
                      local_dir='/content/Geneformer')
    print('base model fetched ->', base_path)
else:
    print('base model already present ->', base_path)

base = BertForMaskedLM.from_pretrained(base_path)

# snapshot a few targeted weights BEFORE merge
watch = [n for n, _ in base.named_parameters()
         if n.endswith('attention.self.query.weight')][:2]
before = {n: base.state_dict()[n].detach().clone() for n in watch}

peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = peft_model.merge_and_unload()
msd = merged.state_dict()

print()
for n in watch:
    d = (msd[n] - before[n]).norm().item()
    bn = before[n].norm().item()
    print(f'{n}\n   base_norm={bn:.4f}  delta_norm={d:.6f}  rel_delta={d/bn:.6e}')

any_move = any((msd[n] - before[n]).norm().item() > 0 for n in watch)
print()
print('VERDICT:', 'merge folds the adapter in -> #3 ruled out' if any_move else 'merge is a no-op -> extractor embedded the base model')

## Stage 3 — plan (only if Stages 1 and 2 both pass)

If the adapter moved and the merge folds it in, the mechanism is sound and the tiny drift is either a genuine null or undertraining. The decisive discriminator is a direct **base-vs-merged embedding delta on a small sample**: re-tokenize a few hundred held-out-test cells, mean-pool-embed them through both the base and the re-merged model, and compute the same per-cell cosine drift the notebook used.

- If sample drift ~ 0.005 (matches §7a): the 0.0056 is **real** — a genuinely tiny effect, not a bug. Decision moves to genuine-null vs undertrained (capacity/steps/LR), which is a methodological call, not a knob to crank.
- If sample drift ~ 0: §6a's specific path did not use the merged weights despite Stage 2 passing — a wiring problem in the extractor handoff.

This stage needs the glia substrate re-tokenized (heavier) so it is left unbuilt until Stages 1-2 justify it.